In [1]:
print("Simpal")

Simpal


In [7]:

import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
from pathlib import Path

HOST = "18.136.157.135"
PORT = 3306
USERNAME = "dm_team5"
PASSWORD = "DM!$!Team!520@4!23&"

engine = create_engine(
    f"mysql+pymysql://{USERNAME}:{quote_plus(PASSWORD)}@{HOST}:{PORT}/",
    connect_args={"connect_timeout": 15}
)

output_file = Path.cwd() / "customer_segmentation_data.csv"

try:
    with engine.connect() as conn:
        databases = [
            row[0] for row in conn.execute(text("SHOW DATABASES"))
        ]

    found = False

    for database in databases:
        if database in {
            "information_schema", "mysql",
            "performance_schema", "sys"
        }:
            continue

        db = database.replace("`", "``")

        try:
            with engine.connect() as conn:
                tables = conn.execute(
                    text(f"SHOW TABLES FROM `{db}`")
                ).fetchall()

            for row in tables:
                table = row[0]
                tbl = table.replace("`", "``")

                try:
                    # Read every column from this table
                    with engine.connect() as conn:
                        df = pd.read_sql_query(
                            f"SELECT * FROM `{db}`.`{tbl}`",
                            conn
                        )

                    # Identify the shopping dataset
                    cols = {
                        str(c).strip().lower()
                        for c in df.columns
                    }

                    if (
                        "age" in cols
                        and "category" in cols
                        and "item purchased" in cols
                        and any("purchase amount" in c for c in cols)
                    ):
                        df.to_csv(output_file, index=False)

                        print("Dataset downloaded successfully!")
                        print("Database:", database)
                        print("Table:", table)
                        print("Total rows:", len(df))
                        print("Total columns:", len(df.columns))
                        print("All columns:", df.columns.tolist())
                        print("Saved to:", output_file)

                        found = True
                        break

                except Exception:
                    continue

            if found:
                break

        except Exception:
            continue

    if not found:
        print("No matching dataset found in accessible tables.")

except Exception as e:
    print("Error:", e)

finally:
    engine.dispose()


C:\Users\DELL\AppData\Local\Temp\ipykernel_28272\98345902.py:48: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(


No matching dataset found in accessible tables.


In [8]:

import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

HOST = "18.136.157.135"
PORT = 3306
USERNAME = "dm_team5"
PASSWORD = "DM!$!Team!520@4!23&"

engine = create_engine(
    f"mysql+pymysql://{USERNAME}:{quote_plus(PASSWORD)}@{HOST}:{PORT}/",
    connect_args={"connect_timeout": 15}
)

results = []

try:
    with engine.connect() as conn:
        databases = [
            row[0] for row in conn.execute(text("SHOW DATABASES"))
        ]

    for database in databases:
        if database in {
            "information_schema", "mysql",
            "performance_schema", "sys"
        }:
            continue

        db = database.replace("`", "``")

        try:
            with engine.connect() as conn:
                tables = conn.execute(
                    text(f"SHOW TABLES FROM `{db}`")
                ).fetchall()

            for row in tables:
                table = row[0]
                tbl = table.replace("`", "``")

                try:
                    with engine.connect() as conn:
                        columns = conn.execute(
                            text(f"SHOW COLUMNS FROM `{db}`.`{tbl}`")
                        ).fetchall()

                    column_names = [column[0] for column in columns]

                    results.append({
                        "Database": database,
                        "Table": table,
                        "Column_Count": len(column_names),
                        "Columns": ", ".join(column_names)
                    })

                except Exception as e:
                    print(f"Could not inspect {database}.{table}: {e}")

        except Exception as e:
            print(f"Could not inspect database {database}: {e}")

    tables_df = pd.DataFrame(results)

    if not tables_df.empty:
        # Show tables likely related to shopping or customer segmentation
        relevant = tables_df[
            tables_df["Database"].str.contains(
                "shop|customer|segment", case=False, na=False
            )
            | tables_df["Table"].str.contains(
                "shop|customer|segment|trend", case=False, na=False
            )
        ]

        print("Potentially relevant tables:")
        display(relevant)

        tables_df.to_csv("available_database_tables.csv", index=False)
        print("\nFull table inventory saved to available_database_tables.csv")
    else:
        print("No accessible tables were found.")

finally:
    engine.dispose()


Potentially relevant tables:


,Database,Table,Column_Count,Columns
1,customer segmentation,Subscribed_customers,18,"CustomerID, Age, Gender, ItemPurchased, Catego..."
2,customer segmentation,customer segmentation,3,"emp_id, name, address"
3,customer segmentation,shopping_trends,18,"CustomerID, Age, Gender, ItemPurchased, Catego..."
4,customer segmentation,shopping_trends_dataset,18,"CustomerID, Age, Gender, ItemPurchased, Catego..."
7,onlinebookstore,Customers,6,"Customer_ID, Name, Email, Phone, City, Country"
15,project_customer_segmentation,customer,10,"invoice_no, customer_id, gender, age, category..."
36,project_shopping_trends,shoping_table,19,"Customer_ID, Age, Gender, Item_Purchased, Cate..."
37,project_shopping_trends,shopping,19,"Customer_ID, Age, Gender, Item_Purchased, Cate..."
38,project_shopping_trends,shopping_table,19,"Customer_ID, Age, Gender, Item_Purchased, Cate..."
39,project_shopping_trends,std,19,"CustomerID, Age, Gender, ItemPurchased, Catego..."



Full table inventory saved to available_database_tables.csv


In [9]:

import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from pathlib import Path

HOST = "18.136.157.135"
PORT = 3306
USERNAME = "dm_team5"
PASSWORD = "DM!$!Team!520@4!23&"

DATABASE = "customer segmentation"
TABLE = "shopping_trends_dataset"

engine = create_engine(
    f"mysql+pymysql://{USERNAME}:{quote_plus(PASSWORD)}"
    f"@{HOST}:{PORT}/{quote_plus(DATABASE)}",
    connect_args={"connect_timeout": 15}
)

try:
    query = f"SELECT * FROM `{TABLE}`"

    df = pd.read_sql_query(query, engine)

    output_file = Path.cwd() / "customer_segmentation_data.csv"
    df.to_csv(output_file, index=False)

    print("Download successful!")
    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Column names:", df.columns.tolist())
    print("File saved at:", output_file)

finally:
    engine.dispose()


C:\Users\DELL\AppData\Local\Temp\ipykernel_28272\647177657.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, engine)


AttributeError: 'Engine' object has no attribute 'cursor'

In [10]:

import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
from pathlib import Path

HOST = "18.136.157.135"
PORT = 3306
USERNAME = "dm_team5"
PASSWORD = "DM!$!Team!520@4!23&"

DATABASE = "customer segmentation"
TABLE = "shopping_trends_dataset"

engine = create_engine(
    f"mysql+pymysql://{USERNAME}:{quote_plus(PASSWORD)}"
    f"@{HOST}:{PORT}/{quote_plus(DATABASE)}",
    connect_args={"connect_timeout": 15}
)

try:
    query = text(f"SELECT * FROM `{TABLE}`")

    # Explicitly open a SQLAlchemy connection
    with engine.connect() as connection:
        df = pd.read_sql_query(query, connection)

    # Save all rows and columns
    output_file = Path.cwd() / "customer_segmentation_data.csv"
    df.to_csv(output_file, index=False)

    print("Download successful!")
    print("Total rows:", len(df))
    print("Total columns:", len(df.columns))
    print("Columns:", df.columns.tolist())
    print("Saved to:", output_file)

except Exception as e:
    print("Error:", repr(e))

finally:
    engine.dispose()


Error: OperationalError('(pymysql.err.OperationalError) (1049, "Unknown database \'customer+segmentation\'")')


In [11]:

import pandas as pd
from sqlalchemy import create_engine, URL, text
from pathlib import Path

HOST = "18.136.157.135"
PORT = 3306
USERNAME = "dm_team5"
PASSWORD = "DM!$!Team!520@4!23&"

DATABASE = "customer segmentation"
TABLE = "shopping_trends_dataset"

# Safely build the database connection URL
connection_url = URL.create(
    drivername="mysql+pymysql",
    username=USERNAME,
    password=PASSWORD,
    host=HOST,
    port=PORT,
    database=DATABASE
)

engine = create_engine(
    connection_url,
    connect_args={"connect_timeout": 15}
)

try:
    with engine.connect() as connection:
        df = pd.read_sql_query(
            text(f"SELECT * FROM `{TABLE}`"),
            connection
        )

    output_file = Path.cwd() / "customer_segmentation_data.csv"
    df.to_csv(output_file, index=False)

    print("Download successful!")
    print("Total rows:", len(df))
    print("Total columns:", len(df.columns))
    print("Columns:", df.columns.tolist())
    print("Saved to:", output_file)

except Exception as e:
    print("Error:", repr(e))

finally:
    engine.dispose()


ImportError: cannot import name 'URL' from 'sqlalchemy' (c:\Users\DELL\AppData\Local\Programs\Python\Python314\Lib\site-packages\sqlalchemy\__init__.py)

In [12]:
import sqlalchemy
print(sqlalchemy.__version__)

1.4.54


In [13]:

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from pathlib import Path

HOST = "18.136.157.135"
PORT = 3306
USERNAME = "dm_team5"
PASSWORD = "DM!$!Team!520@4!23&"

DATABASE = "customer segmentation"
TABLE = "shopping_trends_dataset"

# Safely handle spaces and special characters
connection_url = URL.create(
    "mysql+pymysql",
    username=USERNAME,
    password=PASSWORD,
    host=HOST,
    port=PORT,
    database=DATABASE
)

engine = create_engine(
    connection_url,
    connect_args={"connect_timeout": 15}
)

try:
    with engine.connect() as connection:
        df = pd.read_sql_query(
            text(f"SELECT * FROM `{TABLE}`"),
            connection
        )

    output_file = Path.cwd() / "customer_segmentation_data.csv"
    df.to_csv(output_file, index=False)

    print("Download successful!")
    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Column names:", df.columns.tolist())
    print("Saved to:", output_file)

except Exception as e:
    print("Error:", repr(e))

finally:
    engine.dispose()


Error: TypeError('Query must be a string unless using sqlalchemy.')


C:\Users\DELL\AppData\Local\Temp\ipykernel_28272\2448523131.py:31: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(


In [14]:

import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from pathlib import Path

HOST = "18.136.157.135"
PORT = 3306
USERNAME = "dm_team5"
PASSWORD = "DM!$!Team!520@4!23&"

DATABASE = "customer segmentation"
TABLE = "shopping_trends_dataset"

connection_url = URL.create(
    "mysql+pymysql",
    username=USERNAME,
    password=PASSWORD,
    host=HOST,
    port=PORT,
    database=DATABASE
)

engine = create_engine(
    connection_url,
    connect_args={"connect_timeout": 15}
)

try:
    # Fetch all rows and all columns directly
    with engine.connect() as connection:
        result = connection.exec_driver_sql(
            f"SELECT * FROM `{TABLE}`"
        )

        columns = list(result.keys())
        rows = result.fetchall()

    df = pd.DataFrame(rows, columns=columns)

    output_file = Path.cwd() / "customer_segmentation_data.csv"
    df.to_csv(output_file, index=False, encoding="utf-8-sig")

    print("Download successful!")
    print("Total rows:", len(df))
    print("Total columns:", len(df.columns))
    print("Column names:", df.columns.tolist())
    print("Saved to:", output_file)

except Exception as e:
    print("Error:", repr(e))

finally:
    engine.dispose()


Download successful!
Total rows: 3900
Total columns: 18
Column names: ['CustomerID', 'Age', 'Gender', 'ItemPurchased', 'Category', 'PurchaseAmountUSD', 'Location', 'Size', 'Color', 'Season', 'ReviewRating', 'SubscriptionStatus', 'ShippingType', 'DiscountApplied', 'PromoCodeUsed', 'PreviousPurchases', 'PaymentMethod', 'FrequencyOfPurchases']
Saved to: c:\Users\DELL\Desktop\Customer Sigmentation\customer_segmentation_data.csv
